# Data Preprocessing

## Objective

The objective of this notebook is to preprocess the Household Power Consumption dataset before applying feature engineering and anomaly detection.

The preprocessing pipeline includes:
- Dataset loading and inspection
- Data quality assessment
- Missing value analysis
- Data type conversion
- Temporal consistency verification
- Missing value treatment
- Continuous segment extraction
- Preparation of a clean dataset for downstream machine learning tasks

A properly preprocessed dataset ensures reliable feature engineering, improves model performance, and reduces the impact of noisy or incomplete observations.

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

## 1.Initial Data Inspection

In [ ]:
sample_df=pd.read_csv("data/raw/household_power_consumption.txt",sep=';',nrows=10)
sample_df

In [ ]:
sample_df.columns

In [ ]:
df=pd.read_csv("data/raw/household_power_consumption.txt",sep=';',na_values=["?",""])

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

## 2.Dataset Overview

In [ ]:
df.info()

In [ ]:
df.memory_usage()

In [ ]:
df.memory_usage(deep=True).sum()

In [ ]:
memory_mb=(df.memory_usage(deep=True).sum()/1024**2)
print(f"Dataset Memory Usage:{memory_mb:.2f}MB")

In [ ]:
df.isnull().sum()

## Observation

The dataset contains household electricity consumption measurements collected over time.

Initial inspection helps identify:
- Number of observations
- Available features
- Data types
- Memory usage
- Presence of missing values

This provides a clear understanding of the dataset before applying preprocessing techniques.

## 3.Missing value analysis

In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum()/ len(df))*100
missing_data = pd.DataFrame({"Missing Count": missing_count,"Missing Percentage": missing_percentage})
missing_data

In [ ]:
df.duplicated().sum()

## Observation

Missing values can negatively affect statistical analysis and machine learning models.

This step identifies:
- Columns containing missing values
- Percentage of missing observations
- Overall data completeness

Understanding the missing value distribution helps determine an appropriate preprocessing strategy.

## 4.Data Type Conversion

In [ ]:
numeric_columns = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2","Sub_metering_3"]

In [ ]:
for col in numeric_columns :
    df[col] = pd.to_numeric(df[col],errors='coerce')

In [ ]:
df.dtypes

## Observation

Correct data types improve computational efficiency and ensure mathematical operations are performed accurately.

Converting date and time columns into datetime format enables time-based analysis, while numeric conversion allows statistical computations and visualization.

## 5.Temporal Data Quality Assessment

In [ ]:
df["datetime"] = pd.to_datetime(df["Date"]+ " " + df["Time"],format="%d/%m/%Y %H:%M:%S")

In [ ]:
df[["Date","Time","datetime"]].head()

In [ ]:
df["datetime"].dtype

In [ ]:
print("Start Date:",df["datetime"].min())
print("End Date:",df["datetime"].max())

In [ ]:
df["datetime"].is_monotonic_increasing

In [ ]:
duplicated_timestamps = (df["datetime"].duplicated().sum())
print("Duplicated Timestamps:",duplicated_timestamps)

## Observation

Since this dataset represents a time series, maintaining chronological consistency is essential.

Temporal quality assessment verifies:
- Proper timestamp ordering
- Missing timestamps
- Duplicate records
- Continuity of observations

Reliable temporal information is necessary for detecting energy consumption patterns and anomalies.

## 6.Missing Value Treatment

In [ ]:
df[df.isnull().any(axis=1)].head(20)

In [ ]:
df[df.isnull().any(axis=1)].tail(20)

In [ ]:
df["has_missing"]=(df[numeric_columns].isnull().any(axis=1))
df["has_missing"].value_counts()

In [ ]:
df.drop(columns="has_missing",inplace=True)

In [ ]:
missing_mask = df[numeric_columns].isnull().any(axis=1)
missing_group = (missing_mask != missing_mask.shift()).cumsum()

In [ ]:
missing_blocks = (df[missing_mask].groupby(missing_group).size().sort_values(ascending=False))
missing_blocks.head(10)

In [ ]:
df_cleaned = df.copy()
df_cleaned[numeric_columns] = df_cleaned[numeric_columns].interpolate(method="linear",limit=5,limit_area="inside")
df_cleaned.isnull().sum()

In [ ]:
print("Before Interpolation:")
print(df[numeric_columns].isnull().sum())
print("After Interpolation:")
print(df_cleaned[numeric_columns].isnull().sum())


In [ ]:
model_df = df_cleaned.dropna(subset=numeric_columns).copy()

In [ ]:
model_df[numeric_columns].isnull().sum()

In [ ]:
print("Cleaned Dataset Shape:",model_df.shape)

In [ ]:
removed_rows = len(df_cleaned) - len(model_df)
removed_percentage = (removed_rows/len(df_cleaned))*100
print("Rows Excluded:",removed_rows)
print(f"Percentage Excluded: " f"{removed_percentage:.2f}%")

In [ ]:
df_cleaned = df_cleaned.sort_values("datetime").copy()

df_cleaned["time_gap"] = df_cleaned["datetime"].diff()

df_cleaned["segment_id"] = (
    df_cleaned["time_gap"] > pd.Timedelta(minutes=1)
).cumsum()

print("Number of continuous segments:",
      df_cleaned["segment_id"].nunique())

df_cleaned[
    ["datetime", "time_gap", "segment_id", "Sub_metering_3"]
].head(10)

In [ ]:
print("Missing Sub_metering_3:",
      df_cleaned["Sub_metering_3"].isnull().sum())

print("segment_id exists:",
      "segment_id" in df_cleaned.columns)

print("Number of segments:",
      df_cleaned["segment_id"].nunique())

## Observation

Rather than removing all missing observations, an appropriate treatment strategy preserves as much useful information as possible while maintaining data quality.

This helps reduce information loss and improves the reliability of downstream analysis.

## 7.Missing Block Analysis in Continuous Time Series

In [ ]:
missing_blocks_summary = []

for segment_id, segment in df_cleaned.groupby("segment_id"):

    missing_mask = segment["Sub_metering_3"].isnull()

    missing_group = (missing_mask != missing_mask.shift()).cumsum()

    block_sizes = (
        segment[missing_mask].groupby(missing_group[missing_mask]).size()
    )

    for block_size in block_sizes:

        missing_blocks_summary.append({
            "segment_id": segment_id,
            "block_size": block_size
        })

print("Number of missing blocks:",
      len(missing_blocks_summary))

In [ ]:
missing_blocks_df = pd.DataFrame(missing_blocks_summary)
missing_blocks_df.head(10)

In [ ]:
missing_blocks_df["block_size"].describe()

In [ ]:
missing_blocks_df.sort_values("block_size",ascending=False).head(10)

In [ ]:
model_df = df_cleaned.dropna(subset=numeric_columns).copy()

model_df = model_df.sort_values("datetime").reset_index(drop=True)

In [ ]:
print("Model dataset shape:",model_df.shape)

print("Remaining missing values:")
print(model_df[numeric_columns].isnull().sum())

In [ ]:
model_df["time_gap"] = (model_df["datetime"].diff())

In [ ]:
model_df["segment_id"] = (model_df["time_gap"] > pd.Timedelta(minutes=1)).cumsum()

print("Number of continuous valid segments:",model_df["segment_id"].nunique())

In [ ]:
segment_summary = (model_df.groupby("segment_id").agg(start_time=("datetime", "min"),end_time=("datetime", "max"),num_rows=("datetime", "size")))

segment_summary["duration_hours"] = (segment_summary["end_time"] - segment_summary["start_time"]).dt.total_seconds()/3600

segment_summary.sort_values("num_rows",ascending=True).head(10)

In [ ]:
segment_summary["num_rows"].describe()

## Observation

Machine learning models and rolling statistical features require continuous sequences of observations.

Identifying uninterrupted segments ensures that rolling windows, lag features, and anomaly detection algorithms operate on valid time-series data.

## 8.Removal of Insufficient Continuous Segments

In [ ]:
min_segment_length = 60

valid_segments = segment_summary[segment_summary["num_rows"] >= min_segment_length].index

model_df = model_df[model_df["segment_id"].isin(valid_segments)].copy()

model_df = model_df.reset_index(drop=True)

print("Final model dataset shape:",model_df.shape)

print("Remaining segments:",model_df["segment_id"].nunique())

print("Remaining missing values:",model_df[numeric_columns].isnull().sum().sum())

## Final Remarks

After completing all preprocessing steps:

- Missing values have been appropriately handled.
- Data types have been standardized.
- Temporal consistency has been verified.
- Invalid or incomplete records have been removed.
- Continuous data segments have been identified.
- The resulting dataset is ready for feature engineering and anomaly detection.

This preprocessing stage establishes a reliable foundation for subsequent machine learning tasks.

In [ ]:
output_path="data/processed/cleaned_energy_data.csv"
model_df.to_csv(output_path,index=False)
print("Preprocessed dataset saved to:",output_path)

In [ ]:
print("=" * 50)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 50)

print("Final dataset shape:",
      model_df.shape)

print("Date range:",
      model_df["datetime"].min(),
      "to",
      model_df["datetime"].max())

print("Continuous segments:",
      model_df["segment_id"].nunique())

print("Total missing numerical values:",
      model_df[numeric_columns]
      .isnull()
      .sum()
      .sum())

print("Duplicate rows:",
      model_df.duplicated().sum())

print("=" * 50)

In [ ]:
print(model_df.columns.tolist())

# Conclusion

The preprocessing pipeline successfully transformed the raw household power consumption dataset into a structured, high-quality dataset suitable for machine learning.

Key achievements include:

- Comprehensive dataset inspection
- Missing value identification and treatment
- Data type standardization
- Temporal consistency verification
- Continuous time-series extraction
- Improved data quality for feature engineering

The processed dataset serves as the foundation for feature engineering, anomaly detection using Isolation Forest, and explainable AI analysis using SHAP.